In [180]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score

In [181]:
def charger_immobilier():
    """Charge California Housing, renvoie X, y.
    Affiche de manière robuste le nombre de lignes, de variables et l'unité de la cible.
    """
    data = fetch_california_housing()
    X, y = data.data, data.target
    
    nb_lignes, nb_variables = X.shape
    nom_cible = data.target_names[0] if hasattr(data, 'target_names') else "Cible"
    
    # Extraction robuste de la ligne descriptive de la cible
    unite_cible = "Non trouvée textuellement"
    
    for line in data.DESCR.split('\n'):
        # On cherche des mots-clés universels dans la description du dataset
        if "median house value" in line.lower() or nom_cible.lower() in line.lower():
            if ":" in line: # Souvent formaté comme "Nom : description"
                unite_cible = line.split(":", 1)[1].strip()
            else:
                unite_cible = line.strip()
            break

    # Affichage propre
    print(f"California Housing : ({nb_lignes}, {nb_variables}) variables")
    print(f"Cible ({nom_cible}) : {unite_cible}")
    
    return X, y

In [182]:
X, y = charger_immobilier()
    
# Séparation Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



California Housing : (20640, 8) variables
Cible (MedHouseVal) : The target variable is the median house value for California districts,


In [183]:
def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    """Entraîne, prédit, renvoie un dict {r2, mae, rmse}.
    Doit renvoyer les 3 métriques de régression vues en section 2.
    """
    # Entraînement
    modele.fit(X_train, y_train)
    
    # Prédiction
    y_pred = modele.predict(X_test)
    
    # Calcul des métriques
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    
    return {"r2": r2, "mae": mae, "rmse": rmse}

In [184]:
# Définition des modèles
lr = LinearRegression()
rf = RandomForestRegressor(random_state=42, n_estimators=100)

In [185]:

# ------------------------------------------
# CAS NORMAL : Dataset complet
# ------------------------------------------
print("\n--- CAS NORMAL ---")
res_lr = evaluer_regression(lr, X_train_scaled, X_test_scaled, y_train, y_test)
res_rf = evaluer_regression(rf, X_train_scaled, X_test_scaled, y_train, y_test)

print(f"LinearRegression : R2={res_lr['r2']:.2f} MAE={res_lr['mae']:.2f} RMSE={res_lr['rmse']:.2f}")
print(f"RandomForest     : R2={res_rf['r2']:.2f} MAE={res_rf['mae']:.2f} RMSE={res_rf['rmse']:.2f}")


--- CAS NORMAL ---
LinearRegression : R2=0.58 MAE=0.53 RMSE=0.75
RandomForest     : R2=0.80 MAE=0.33 RMSE=0.51


In [186]:
# ------------------------------------------
# CAS LIMITE : Entraînement sur 100 lignes
# ------------------------------------------
print("\n--- CAS LIMITE (100 lignes) ---")
X_train_lim, y_train_lim = X_train_scaled[:100], y_train[:100]

res_lr_lim = evaluer_regression(lr, X_train_lim, X_test_scaled, y_train_lim, y_test)
res_rf_lim = evaluer_regression(rf, X_train_lim, X_test_scaled, y_train_lim, y_test)

print(f"LinearRegression (Limité) : R2={res_lr_lim['r2']:.2f} MAE={res_lr_lim['mae']:.2f} RMSE={res_lr_lim['rmse']:.2f}")
print(f"RandomForest (Limité)     : R2={res_rf_lim['r2']:.2f} MAE={res_rf_lim['mae']:.2f} RMSE={res_rf_lim['rmse']:.2f}")


--- CAS LIMITE (100 lignes) ---
LinearRegression (Limité) : R2=0.40 MAE=0.54 RMSE=0.88
RandomForest (Limité)     : R2=0.54 MAE=0.56 RMSE=0.77


In [187]:
# ------------------------------------------
# CAS ADVERSARIAL : Quartier fictif hors-norme
# ------------------------------------------
print("\n--- CAS ADVERSARIAL ---")
valeurs_moyennes = np.mean(X, axis=0)
quartier_fictif = valeurs_moyennes.copy()
quartier_fictif[0] = 0.0    # Revenu médian à 0
quartier_fictif[4] = 9000.0 # Population énorme

quartier_fictif_scaled = scaler.transform([quartier_fictif])

pred_lr = lr.predict(quartier_fictif_scaled)[0]
pred_rf = rf.predict(quartier_fictif_scaled)[0]

print(f"Prédiction LinearRegression : {pred_lr:.2f} (soit {pred_lr*100000:.0f} $)")
print(f"Prédiction RandomForest     : {pred_rf:.2f} (soit {pred_rf*100000:.0f} $)")


--- CAS ADVERSARIAL ---
Prédiction LinearRegression : 0.17 (soit 16913 $)
Prédiction RandomForest     : 1.19 (soit 119458 $)


In [188]:
def charger_airbnb(url_csv):
    """Charge le CSV, garde les colonnes numériques utiles, nettoie les NaN.

    Doit renvoyer un DataFrame propre, sans valeurs manquantes.
    """
    # 1. Chargement des données
    df = pd.read_csv(url_csv)

    # 2. Sélection des colonnes numériques pertinentes
    colonnes_utiles = [
        "price",
        "minimum_nights",
        "number_of_reviews",
        "availability_365",
    ]

    # Sécurité au cas où les noms de colonnes varient légèrement selon le dataset d'Airbnb
    colonnes_presentes = [col for col in colonnes_utiles if col in df.columns]
    df_numeric = df[colonnes_presentes]

    df
    # 3. Gestion des NaN (Imputation par la médiane pour ne pas perdre de lignes, ou dropna)
    # Ici, on opte pour dropna pour avoir des données 100% réelles, ou fillna pour la robustesse.
    df_clean = df_numeric.dropna().copy()
    df_clean["price"] = df_clean["price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)
    print(
        f"Listings chargés : {df_clean.shape[0]} lignes, {df_clean.shape[1]} colonnes numériques retenues"
    )
    return df_clean

In [189]:
df_airbnb = charger_airbnb("listings.csv")

Listings chargés : 5256 lignes, 4 colonnes numériques retenues


In [190]:
def choisir_k(X_scaled, k_range=range(2, 15)):
    """Pour chaque k, renvoie inertie et silhouette.

    Doit afficher un tableau permettant de repérer le coude / le meilleur k.
    """
    print(f"{'k':<5} | {'Inertie':<12} | {'Silhouette':<10}")
    print("-" * 35)

    for k in k_range:
        # Initialisation de KMeans (n_init=10 explicite pour éviter les warnings)
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
        cluster_labels = kmeans.fit_predict(X_scaled)

        # Récupération des métriques
        inertia = kmeans.inertia_
        sil = silhouette_score(X_scaled, cluster_labels)

        # Affichage formaté
        print(f"k={k:<3} | inertie={inertia:<10.1f} | silhouette={sil:.2f}")

In [191]:
# Standardisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_airbnb)

choisir_k(X_scaled)

k     | Inertie      | Silhouette
-----------------------------------
k=2   | inertie=15864.0    | silhouette=0.76
k=3   | inertie=11647.4    | silhouette=0.38
k=4   | inertie=8484.4     | silhouette=0.43
k=5   | inertie=6053.0     | silhouette=0.46
k=6   | inertie=4946.9     | silhouette=0.47
k=7   | inertie=4183.0     | silhouette=0.47
k=8   | inertie=3805.5     | silhouette=0.34
k=9   | inertie=3471.1     | silhouette=0.35
k=10  | inertie=3166.1     | silhouette=0.35
k=11  | inertie=2871.4     | silhouette=0.36
k=12  | inertie=2613.2     | silhouette=0.36
k=13  | inertie=2459.7     | silhouette=0.32
k=14  | inertie=2266.2     | silhouette=0.32


In [192]:
print("--- 1. CAS NORMAL ---")


# Entraînement du modèle final
k_choisi = 6
kmeans_normal = KMeans(n_clusters=k_choisi, n_init=10, random_state=42)
df_airbnb["cluster_normal"] = kmeans_normal.fit_predict(X_scaled)

# Description des segments basée sur les moyennes des vraies données
print("\nProfil des segments (Cas Normal) :")
print(df_airbnb.groupby("cluster_normal").mean())



--- 1. CAS NORMAL ---

Profil des segments (Cas Normal) :
                      price  minimum_nights  number_of_reviews  \
cluster_normal                                                   
0                 87.897941        5.897065          40.315375   
1                130.470588      364.235294          23.223529   
2                 86.270959        3.525243          40.980083   
3                 81.528217        2.004515         381.613995   
4               1213.111111        3.833333          17.611111   
5                340.044776        2.992537          36.660448   

                availability_365  
cluster_normal                    
0                     300.604906  
1                     333.682353  
2                      77.735063  
3                     208.316027  
4                     288.000000  
5                     231.690299  


Cluster 0 : Les "Dispo Long Terme / Standard"

Profil : Prix modéré (88 €), séjours d'environ une semaine (~6 nuits), moyennement commentés, mais disponibles presque toute l'année (300 jours).

Cluster 1 : Les "Baux Commerciaux / Professionnels"

Profil : Ce groupe est très marqué par un minimum_nights à 364 nuits. Ce ne sont pas des locations de vacances, mais des appartements meublés loués à l'année (étudiants, professionnels en mobilité).

Cluster 2 : Les "Occasionnels / Locaux"

Profil : Prix abordable (86 €), séjours courts (3-4 nuits). La particularité est leur très faible disponibilité (77 jours par an). Ce sont probablement des locaux qui louent leur propre logement uniquement pendant leurs vacances.

Cluster 3 : Les "Stars de la plateforme / Ultra-populaires"

Profil : Un nombre d'avis gigantesque (381 avis en moyenne) pour un prix attractif (81 €) et des séjours courts (2 nuits). Ce sont des logements à forte rotation (ex: petits studios hyper-centraux ou proches des gares) gérés de manière industrielle.

Cluster 4 : Le "Grand Luxe / Ultra-Premium"

Profil : Prix astronomique (1 213 € la nuit en moyenne), peu d'avis. Ce segment regroupe les villas d'exception ou les penthouses de luxe.

Cluster 5 : Le "Premium Standard"

Profil : Logements haut de gamme mais accessibles (340 € la nuit), loués pour des week-ends ou courts séjours (3 nuits).

In [193]:
# =====================================================================
# 2. CAS LIMITE : SANS STANDARDISATION
# =====================================================================
print("\n--- 2. CAS LIMITE (SANS STANDARDISATION) ---")
kmeans_sans_scale = KMeans(n_clusters=k_choisi, n_init=10, random_state=42)
df_airbnb["cluster_sans_scale"] = kmeans_sans_scale.fit_predict(df_airbnb[
    ["price", "minimum_nights", "number_of_reviews", "availability_365"]
])

print("\nProfil des segments (Sans Standardisation) :")
print(df_airbnb.groupby("cluster_sans_scale").mean())




--- 2. CAS LIMITE (SANS STANDARDISATION) ---

Profil des segments (Sans Standardisation) :
                          price  minimum_nights  number_of_reviews  \
cluster_sans_scale                                                   
0                    349.486056       15.976096          34.366534   
1                     89.239051       17.437500          29.093522   
2                     84.320652        2.882246         241.692029   
3                   1213.111111        3.833333          17.611111   
4                     80.922535        4.154930         578.767606   
5                     87.529748        4.431223          34.922418   

                    availability_365  cluster_normal  
cluster_sans_scale                                    
0                         242.462151        4.856574  
1                         304.424726        0.045164  
2                         214.331522        1.980072  
3                         288.000000        4.000000  
4                

In [194]:
# =====================================================================
# 3. CAS ADVERSARIAL : INJECTION D'UN OUTLIER
# =====================================================================
print("\n--- 3. CAS ADVERSARIAL (VALEUR ABERRANTE) ---")

# On crée une copie des données chargées 
df_adversarial = df_airbnb[
    ["price", "minimum_nights", "number_of_reviews", "availability_365"]
].copy()

# On lance le KMeans
kmeans_adv = KMeans(n_clusters=k_choisi, n_init=10, random_state=42)
df_adversarial["cluster_adv"] = kmeans_adv.fit_predict(X_scaled)

print("\nTaille des clusters avant injection de l'outlier :")
print(df_adversarial["cluster_adv"].value_counts())

df_adversarial_fake = df_airbnb[
    ["price", "minimum_nights", "number_of_reviews", "availability_365"]
].copy()
# on injecte l'annonce à 100 000 €
annonce_fake = pd.DataFrame(
    [{
        "price": 100000.0,
        "minimum_nights": 1,
        "number_of_reviews": 0,
        "availability_365": 365,
    }]
)
df_adversarial_fake = pd.concat([df_adversarial_fake, annonce_fake], ignore_index=True)

# On doit re-standardiser ce nouveau dataset pollué
X_adv_scaled = scaler.fit_transform(df_adversarial_fake)

# On relance le KMeans
kmeans_adv = KMeans(n_clusters=k_choisi, n_init=10, random_state=42)
df_adversarial_fake["cluster_adv"] = kmeans_adv.fit_predict(X_adv_scaled)

print("\nTaille des clusters après injection de l'outlier :")
print(df_adversarial_fake["cluster_adv"].value_counts())




--- 3. CAS ADVERSARIAL (VALEUR ABERRANTE) ---

Taille des clusters avant injection de l'outlier :
cluster_adv
0    2283
2    2159
3     443
5     268
1      85
4      18
Name: count, dtype: int64

Taille des clusters après injection de l'outlier :
cluster_adv
4    2285
1    2182
0     563
5     141
2      85
3       1
Name: count, dtype: int64


In [195]:
print("\nProfil des centres de clusters normaux :")
print(df_adversarial.groupby("cluster_adv").mean())
print("\nProfil des centres de clusters pollués (Données brutes) :")
print(df_adversarial_fake.groupby("cluster_adv").mean())


Profil des centres de clusters normaux :
                   price  minimum_nights  number_of_reviews  availability_365
cluster_adv                                                                  
0              87.897941        5.897065          40.315375        300.604906
1             130.470588      364.235294          23.223529        333.682353
2              86.270959        3.525243          40.980083         77.735063
3              81.528217        2.004515         381.613995        208.316027
4            1213.111111        3.833333          17.611111        288.000000
5             340.044776        2.992537          36.660448        231.690299

Profil des centres de clusters pollués (Données brutes) :
                     price  minimum_nights  number_of_reviews  \
cluster_adv                                                     
0                88.916519        2.209591         241.120782   
1                99.367553        3.531164          34.458295   
2              

In [196]:
def charger_donnees_uci():
    txt_path = "SMSSpamCollection"
      
    # Lecture du fichier (séparateur TSV, pas d'en-tête)
    df = pd.read_csv(txt_path, sep='\t', names=['label', 'message'])
    # Encodage de la cible : 1 pour spam, 0 pour ham (normal)
    df['target'] = df['label'].map({'ham': 0, 'spam': 1})
    return df

In [197]:
df = charger_donnees_uci()

In [198]:
def vectoriser_textes(messages, vectorizer=None):
    """Transforme une liste de textes en matrice numérique.
    Si vectorizer est None, en crée un (fit_transform) ; sinon réutilise (transform).
    Renvoie (X, vectorizer).
    """
    if vectorizer is None:
        # On utilise TfidfVectorizer avec des hyperparamètres standards adaptés
        vectorizer = TfidfVectorizer(stop_words='english', min_df=2)
        X = vectorizer.fit_transform(messages)
    else:
        X = vectorizer.transform(messages)
    return X, vectorizer

In [199]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['message'], df['target'], test_size=0.2, random_state=42, stratify=df['target']
)
# Vectorisation
X_train, vectorizer = vectoriser_textes(X_train_raw)
X_test,_ = vectoriser_textes(X_test_raw, vectorizer=vectorizer)

# Vérification concrète
print("=== VÉRIFICATION DES SPAMS DANS LES JEUX ===")
print(f"Nombre de spams dans le Train : {sum(y_train)} (soit {sum(y_train)/len(y_train):.2%})")
print(f"Nombre de spams dans le Test  : {sum(y_test)} (soit {sum(y_test)/len(y_test):.2%})")

=== VÉRIFICATION DES SPAMS DANS LES JEUX ===
Nombre de spams dans le Train : 598 (soit 13.42%)
Nombre de spams dans le Test  : 149 (soit 13.36%)


In [200]:
def evaluer_spam(modele, X_train, X_test, y_train, y_test):
    """Entraîne, prédit, affiche le rapport de classification complet."""
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    
    # Affichage propre du rapport de classification avec les noms explicites
    print(classification_report(y_test, y_pred, target_names=['normal', 'spam']))
    return modele

In [201]:
print("\n=== ÉVALUATION : NAIVE BAYES (Baseline) ===")
nb_model = MultinomialNB()
evaluer_spam(nb_model, X_train, X_test, y_train, y_test)




=== ÉVALUATION : NAIVE BAYES (Baseline) ===
              precision    recall  f1-score   support

      normal       0.97      1.00      0.99       966
        spam       1.00      0.81      0.89       149

    accuracy                           0.97      1115
   macro avg       0.99      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115



,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3859., 598.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.14,-2.01]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 3376)","[[ 0. , 0. , 0. ,...,17.34, 0. , 0.39], [ 2.38, 5.47, 0.48,..., 0. , 1.22, 0.34]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 3376)","[[-9.39,-9.39,-9.39,...,-6.48,-9.39,-9.06], [-7.39,-6.74,-8.22,...,-8.61,-7.81,-8.32]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,3376


In [202]:
print("\n=== ÉVALUATION : RÉGRESSION LOGISTIQUE ===")
lr_model = LogisticRegression(class_weight='balanced', random_state=42)
evaluer_spam(lr_model, X_train, X_test, y_train, y_test)




=== ÉVALUATION : RÉGRESSION LOGISTIQUE ===
              precision    recall  f1-score   support

      normal       0.99      0.98      0.99       966
        spam       0.90      0.91      0.91       149

    accuracy                           0.97      1115
   macro avg       0.94      0.95      0.95      1115
weighted avg       0.98      0.97      0.97      1115



,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default

In [203]:
print("TEST DES SCÉNARIOS SPÉCIFIQUES")

# Scénario Edge Case : Message vide
# Scénario Adversarial : Spam déguisé
tests = {
    "Edge Case (Vide)": "",
    "Adversarial (Déguisé)": "Hi, your package is waiting for you, please confirm here."
}

for nom, texte in tests.items():
    print(f"\n--> {nom} : '{texte}'")
    # Transformation du texte brut via le vectorizer entraîné
    X_sample, vectorizer = vectoriser_textes([texte], vectorizer=vectorizer)
    
    pred_nb = nb_model.predict(X_sample)[0]
    pred_lr = lr_model.predict(X_sample)[0]
    
    label_map = {0: "normal", 1: "spam"}
    print(f"    Prédiction Naive Bayes      : {label_map[pred_nb]}")
    print(f"    Prédiction Régression Logst. : {label_map[pred_lr]}")

TEST DES SCÉNARIOS SPÉCIFIQUES

--> Edge Case (Vide) : ''
    Prédiction Naive Bayes      : normal
    Prédiction Régression Logst. : normal

--> Adversarial (Déguisé) : 'Hi, your package is waiting for you, please confirm here.'
    Prédiction Naive Bayes      : normal
    Prédiction Régression Logst. : normal


In [204]:
def charger_sonar(url):
    df = pd.read_csv(url, header=None)

    X = df.iloc[:, :-1]
    y = df.iloc[:, -1].map({"M": 1, "R": 0})

    mines = (y == 1).sum()
    rochers = (y == 0).sum()

    print(f"Sonar : {X.shape}, mines={mines}, rochers={rochers}")

    return X, y

In [205]:
url = "sonar.all-data"

X, y = charger_sonar(url)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


Sonar : (208, 60), mines=111, rochers=97


In [206]:
modeles = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000))
    ]),
    "SVC (rbf)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf"))
    ]),
    "RandomForest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            random_state=42
        ))
    ])
}


In [207]:
print("\n--- Avec standardisation ---")

for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{nom} : accuracy={acc:.2f}")

modeles_sans_scaling = {
    "LogisticRegression": LogisticRegression(max_iter=5000),
    "SVC (rbf)": SVC(kernel="rbf"),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42
    )
}


--- Avec standardisation ---
LogisticRegression : accuracy=0.81
SVC (rbf) : accuracy=0.81
RandomForest : accuracy=0.83


Analyse

Avec standardisation

La régression logistique et le SVM RBF obtiennent généralement leurs meilleurs résultats.
Le SVM est souvent le plus performant sur Sonar car il y a peu d'exemples (208) mais beaucoup de variables (60).

In [208]:
print("\n--- Sans standardisation ---")

for nom, modele in modeles_sans_scaling.items():
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{nom} : accuracy={acc:.2f}")

echo_panne = pd.DataFrame([[0.0] * 60], columns=X.columns)




--- Sans standardisation ---
LogisticRegression : accuracy=0.79
SVC (rbf) : accuracy=0.79
RandomForest : accuracy=0.83


Sans standardisation

La régression logistique et le SVM peuvent perdre en performance car leurs calculs dépendent des distances et des coefficients.
Le Random Forest est généralement peu affecté car les arbres de décision ne sont pas sensibles à l'échelle des variables.

In [209]:
print("\n--- Cas adversarial : capteur en panne ---")

for nom, modele in modeles.items():
    prediction = modele.predict(echo_panne)[0]

    if hasattr(modele, "predict_proba"):
        confiance = modele.predict_proba(echo_panne).max()
    else:
        confiance = None

    classe = "Mine" if prediction == 1 else "Rocher"

    if confiance is not None:
        print(f"{nom} : {classe} (confiance={confiance:.3f})")
    else:
        print(f"{nom} : {classe}")


--- Cas adversarial : capteur en panne ---
LogisticRegression : Rocher (confiance=1.000)
SVC (rbf) : Rocher
RandomForest : Rocher (confiance=0.683)


Cas adversarial : 60 valeurs à zéro

Tous les modèles renverront quand même une classe.
Une forte confiance ne signifie pas que la prédiction est fiable.
En pratique, un sous-marin ne devrait pas utiliser directement cette prédiction.
Il faudrait détecter en amont les signaux invalides : valeurs toutes nulles, capteur défaillant, signal trop faible, données hors domaine d'entraînement, anomalies ou absence d'écho exploitable.

In [210]:
def fight_des_ia(X_train, X_test, y_train, y_test, metrique):
    """
    Entraîne plusieurs algorithmes sur le même split et affiche un classement.

    Parameters
    ----------
    X_train, X_test : données
    y_train, y_test : labels
    metrique : fonction(y_true, y_pred) -> float
    """

    competiteurs = {
        "LogisticRegression": LogisticRegression(max_iter=5000),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=42
        ),
        "SVC_rbf": SVC(kernel="rbf")
    }

    resultats = []

    for nom, modele in competiteurs.items():

        debut = time.perf_counter()

        modele.fit(X_train, y_train)

        fin = time.perf_counter()

        temps_entrainement = fin - debut

        temps_pred_debut = time.perf_counter()
        y_pred = modele.predict(X_test)
        temps_pred_fin = time.perf_counter()
        temps_prediction = temps_pred_fin - temps_pred_debut

        score = metrique(y_test, y_pred)

        resultats.append({
            "Algo": nom,
            "Score": score,
            "Temps (s)": temps_entrainement,
            "Temps prédiction (s)": temps_prediction
        })

    leaderboard = pd.DataFrame(resultats)

    leaderboard = leaderboard.sort_values(
        by="Score",
        ascending=False
    ).reset_index(drop=True)

    print("\n=== LEADERBOARD ===")

    # for i, row in leaderboard.iterrows():
    #     print(
    #         f"{i+1}. {row['Algo']} : "
    #         f"Score={row['Score']:.4f} "
    #         f"({row['Temps (s)']:.3f}s)"
    #     )

    return leaderboard

In [211]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leaderboard = fight_des_ia(
    X_train,
    X_test,
    y_train,
    y_test,
    metrique=f1_score
)

print(leaderboard)


=== LEADERBOARD ===
                 Algo     Score  Temps (s)  Temps prédiction (s)
0    GradientBoosting  0.863636   0.416824              0.002063
1             SVC_rbf  0.857143   0.004491              0.002923
2        DecisionTree  0.857143   0.006654              0.004486
3        RandomForest  0.844444   0.488351              0.029900
4  LogisticRegression  0.826087   0.028010              0.002867


Le meilleur score est obtenu par GradientBoosting avec un F1-score de 0.8636. Cependant, son temps d'entraînement est significativement plus élevé que celui des autres modèles. Le SVC obtient un score très proche (0.8571) tout en étant environ 75 fois plus rapide à entraîner. Le choix du meilleur modèle dépend donc du compromis entre performance et coût de calcul. Dans notre cas, le SVC apparaît comme le meilleur compromis, tandis que GradientBoosting est le meilleur modèle en termes de performance brute.